# Pharmacophore modeling and screening

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yboulaamane/comp_chem_colab/blob/main/pharmacophore_modeling.ipynb)

A pharmacophore captures *what* a molecule needs to bind, stripped of the particular atoms that
provide it: a hydrogen-bond donor here, an aromatic ring there, a hydrophobic group over there.
Two very different-looking molecules can share one, which is exactly why it is powerful for
finding new chemistry. You'll build a pharmacophore from a known MAO-B inhibitor (safinamide) and
use it to rank compounds two ways.

**What you'll do**

- Identify and visualise the pharmacophore features of the reference molecule.
- Screen a set with a fast **2D pharmacophore fingerprint** (feature pairs and their distances).
- Screen it again with a **3D shape-and-feature overlay** that aligns each molecule onto the reference.
- Score both against real MAO-B actives and inactives from ChEMBL (ROC AUC and enrichment factor).

Runtime: CPU. The 3D overlay takes a couple of minutes.

## Setup

Everything is in RDKit; pandas, scikit-learn and seaborn are on Colab.

In [ ]:
%pip install -q rdkit seaborn

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk", palette="muted")
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titleweight": "bold",
    "axes.titlesize": 16,
    "axes.titlepad": 14,
    "savefig.bbox": "tight",
})

import os
from collections import Counter

from rdkit import Chem, DataStructs, RDConfig, RDLogger
from rdkit.Chem import AllChem, ChemicalFeatures, Draw, rdDistGeom, rdShapeAlign
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit.Chem.Pharm2D import Gobbi_Pharm2D, Generate
from sklearn.metrics import roc_auc_score, RocCurveDisplay

RDLogger.DisableLog("rdApp.*")

## Features of the reference molecule

RDKit's feature factory assigns pharmacophore features from the same definitions used in its
Gobbi 2D fingerprints. Below they are found on safinamide and highlighted on the 2D drawing.

In [ ]:
factory = ChemicalFeatures.BuildFeatureFactory(
    os.path.join(RDConfig.RDDataDir, "BaseFeatures.fdef"))

reference = Chem.MolFromSmiles("C[C@H](NCc1ccc(OCc2cccc(F)c2)cc1)C(N)=O")
features = factory.GetFeaturesForMol(reference)
print("Features found:", dict(Counter(f.GetFamily() for f in features)))

In [ ]:
families = ["Donor", "Acceptor", "Aromatic", "Hydrophobe", "PosIonizable", "NegIonizable"]
colors = dict(zip(families, [(0.2, 0.6, 1), (1, 0.4, 0.4), (0.5, 0.8, 0.3),
                             (1, 0.8, 0.2), (0.7, 0.4, 1), (1, 0.5, 0.8)]))
highlight, hl_colors = {}, {}
for feat in features:
    color = colors.get(feat.GetFamily())
    if color:
        for atom in feat.GetAtomIds():
            highlight.setdefault(atom, color)
drawer = rdMolDraw2D.MolDraw2DCairo(500, 350)
rdMolDraw2D.PrepareAndDrawMolecule(drawer, reference, highlightAtoms=list(highlight),
                                   highlightAtomColors=highlight)
drawer.FinishDrawing()
from IPython.display import Image
Image(drawer.GetDrawingText())

Blue = donor, red = acceptor, green = aromatic, yellow = hydrophobe, purple = positively
ionizable. This pattern (a donor and acceptor near an aromatic ring, with a lipophilic tail)
is what the screens below look for.

## A set to screen: MAO-B actives and inactives from ChEMBL

## Data: MAO-B inhibitors from ChEMBL

The compounds and their activities come from [ChEMBL](https://www.ebi.ac.uk/chembl/)
target [CHEMBL2039](https://www.ebi.ac.uk/chembl/explore/target/CHEMBL2039),
human monoamine oxidase B. The cell below downloads every activity record for
that target through the ChEMBL web services and turns them into one labelled row
per compound:

* Only IC50 and Ki values in nM that ChEMBL has not flagged as doubtful are used.
* Salts and counter-ions are stripped, so the different salt forms of one compound count once.
* A compound is **active (1)** if its median value is at most 316 nM (pIC50 &ge; 6.5) and
  **inactive (0)** if it is at least 10 &micro;M (pIC50 &le; 5). A compound reported only as
  "> 10 &micro;M" is also inactive. Compounds in between are left out.

The labelled table is saved to `maob_chembl_labelled.csv` and later runs read that file
instead of downloading again. Delete it to refresh the data. ChEMBL data is licensed under
[CC BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0/).

In [ ]:
TARGET = "CHEMBL2039"   # monoamine oxidase B, Homo sapiens
ACTIVE_NM = 316         # median IC50 or Ki at or below this: active   (pIC50 >= 6.5)
INACTIVE_NM = 10_000    # median IC50 or Ki at or above this: inactive (pIC50 <= 5)
DATA_FILE = "maob_chembl_labelled.csv"

In [ ]:
import os
import time
from urllib.parse import urljoin

import numpy as np
import pandas as pd
import requests
from rdkit import Chem, RDLogger
from rdkit.Chem.MolStandardize import rdMolStandardize

RDLogger.DisableLog("rdApp.*")
CHEMBL_URL = "https://www.ebi.ac.uk"


def fetch_activities(target):
    """Every activity record ChEMBL holds for one target, 1000 per request."""
    url = f"{CHEMBL_URL}/chembl/api/data/activity.json"
    params = {"target_chembl_id": target, "limit": 1000}
    records = []
    while url:
        for attempt in range(5):
            try:
                response = requests.get(url, params=params, timeout=120)
                response.raise_for_status()
                break
            except requests.RequestException:
                if attempt == 4:
                    raise
                time.sleep(2 ** attempt)
        page = response.json()
        records += page["activities"]
        print(f"\rDownloaded {len(records)} of {page['page_meta']['total_count']} records", end="")
        next_page = page["page_meta"]["next"]
        url, params = (urljoin(CHEMBL_URL, next_page), None) if next_page else (None, None)
    print()
    if not records:
        raise ValueError(f"ChEMBL returned no activity records for {target}")
    return pd.DataFrame(records)


def parent_smiles(smiles):
    """Canonical SMILES of the largest fragment, neutralised, or None if RDKit cannot read it."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    try:
        return Chem.MolToSmiles(rdMolStandardize.ChargeParent(mol))
    except Exception:
        return None


def label_molecules(activities):
    """One row per parent structure: its first ChEMBL ID, SMILES and label (1 active, 0 inactive)."""
    a = activities.copy()
    a["relation"] = a["standard_relation"].fillna("").str.strip("'")
    a["value"] = pd.to_numeric(a["standard_value"], errors="coerce")
    usable = (a["standard_type"].isin(["IC50", "Ki"]) & (a["standard_units"] == "nM")
              & a["data_validity_comment"].isna() & a["canonical_smiles"].notna()
              & (a["value"] > 0))
    a = a[usable].copy()
    a["smiles"] = a["canonical_smiles"].map(parent_smiles)
    a = a[a["smiles"].notna()]

    median = a[a["relation"] == "="].groupby("smiles")["value"].median()
    labels = pd.Series(np.nan, index=median.index)
    labels[median <= ACTIVE_NM] = 1
    labels[median >= INACTIVE_NM] = 0

    # Compounds with no exact value, only "> x" with x of at least INACTIVE_NM
    above = a[a["relation"].isin([">", ">="]) & (a["value"] >= INACTIVE_NM)]["smiles"]
    only_above = pd.Index(above.unique()).difference(median.index)
    labels = pd.concat([labels, pd.Series(0.0, index=only_above)]).dropna().astype(int)

    first_id = a.sort_values("molecule_chembl_id").groupby("smiles")["molecule_chembl_id"].first()
    data = pd.DataFrame({"molecule_chembl_id": first_id[labels.index].to_numpy(),
                         "smiles": labels.index.to_numpy(),
                         "label": labels.to_numpy()})
    return data.sort_values("molecule_chembl_id", ignore_index=True)

In [ ]:
if os.path.exists(DATA_FILE):
    data = pd.read_csv(DATA_FILE)
else:
    data = label_molecules(fetch_activities(TARGET))
    data.to_csv(DATA_FILE, index=False)

n_active = int(data["label"].sum())
print(f"{len(data)} compounds: {n_active} active, {len(data) - n_active} inactive")
data.head()

## 2D pharmacophore fingerprint search

The Gobbi 2D fingerprint encodes pairs of features and the number of bonds between them.
Ranking the set by Tanimoto similarity to the reference is a fast first screen.

In [ ]:
def pharm2d(mol):
    return Generate.Gen2DFingerprint(mol, Gobbi_Pharm2D.factory)


ref_fp = pharm2d(reference)
mols = [Chem.MolFromSmiles(s) for s in data["smiles"]]
data["sim_2d"] = [DataStructs.TanimotoSimilarity(ref_fp, pharm2d(m)) for m in mols]
auc_2d = roc_auc_score(data["label"], data["sim_2d"])
print(f"2D pharmacophore similarity ROC AUC: {auc_2d:.3f}")

## 3D shape-and-feature overlay

This embeds a few conformers of each molecule, overlays each on the reference and keeps the
best combined shape-and-feature (colour) score. It is slower but uses the 3D arrangement of
the features, not just their topology.

In [ ]:
params = rdDistGeom.ETKDGv3()
params.randomSeed = 42
ref3d = Chem.AddHs(reference)
rdDistGeom.EmbedMolecule(ref3d, params)
ref_shape = rdShapeAlign.PrepareConformer(ref3d)


def overlay_score(smiles, n_confs=5):
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    conf_ids = rdDistGeom.EmbedMultipleConfs(mol, n_confs, params)
    if not conf_ids:
        return 0.0
    best = 0.0
    for cid in conf_ids:
        shape, color = rdShapeAlign.AlignMol(ref_shape, mol, probeConfId=cid, opt_param=0.5)
        best = max(best, 0.5 * shape + 0.5 * color)
    return best


data["sim_3d"] = [overlay_score(s) for s in data["smiles"]]
auc_3d = roc_auc_score(data["label"], data["sim_3d"])
print(f"3D overlay ROC AUC: {auc_3d:.3f}")

## How well does each screen enrich the actives?

ROC curves for both scores. The dashed line is random. Enrichment factor at 5 % is the share
of actives caught in the top-scoring 5 % of the set, divided by their overall share, so 1 is
no better than random.

In [ ]:
def enrichment_factor(labels, scores, fraction=0.05):
    order = np.argsort(scores)[::-1]
    n_top = max(1, int(len(scores) * fraction))
    top_rate = labels.to_numpy()[order][:n_top].mean()
    return top_rate / labels.mean()


fig, ax = plt.subplots(figsize=(7, 7))
for column, name in [("sim_2d", "2D fingerprint"), ("sim_3d", "3D overlay")]:
    ef = enrichment_factor(data["label"], data[column])
    RocCurveDisplay.from_predictions(data["label"], data[column], ax=ax,
                                     name=f"{name} (EF 5% = {ef:.1f})")
ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="random")
ax.set(title="Pharmacophore screening: actives vs inactives",
       xlabel="False positive rate", ylabel="True positive rate")
ax.legend(loc="lower right", frameon=True)
plt.tight_layout()
plt.show()

## Takeaways

- A pharmacophore lets a chemically different molecule score well as long as it presents the same
  features in the same places, which is how these methods jump to new scaffolds that a plain
  fingerprint would miss.
- 2D is nearly free and a fine first pass; the 3D overlay is slower but uses the real geometry of
  the features, and usually enriches better when conformers cooperate.
- This model is just one molecule's features. A stronger one aligns several known actives and
  keeps what they share, or reads the features straight from the protein-ligand complex.
- Think of it as a fast filter to shrink a big library before docking, not a replacement for it.

**Try it yourself:** build the reference from a different inhibitor, or raise the conformer count
in `overlay_score` and watch the enrichment move.